# Variant C — Multi-Task Learning (Classify + Explain)

**Architecture:** DeBERTa-v3-base with dual heads:
1. Classification head: NLI label prediction from [CLS]
2. MLM head: masked token prediction on explanation tokens only

**Joint Loss:** `0.7 × CE_label + 0.3 × MLM_explanation`

At inference, the MLM head is discarded.

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers datasets accelerate

In [ ]:
# Cell 2: Mount Google Drive and set up paths
from google.colab import drive
drive.mount("/content/drive")

import os, sys

# ---- CONFIGURE THESE PATHS ----
# Option A: If you cloned the repo to Drive
PROJECT_ROOT = "/content/drive/MyDrive/Old-Explanations-New-Rationales-Bridging-e-SNLI-and-LLM-Chain-of-Thought-in-Encoder-Based-NLI"

# Option B: If you uploaded the repo to Colab
# PROJECT_ROOT = "/content/Old-Explanations-New-Rationales-Bridging-e-SNLI-and-LLM-Chain-of-Thought-in-Encoder-Based-NLI"

sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")

In [ ]:
# Cell 3: Load config and build datasets
from training.config import VariantCConfig
from models.common import load_tokenizer
from data.preprocess import load_esnli_train, load_esnli_split, ESNLIMultiTaskDataset

config = VariantCConfig()

tokenizer = load_tokenizer(config.model_name)
print(f"Tokenizer: {type(tokenizer).__name__} (is_fast={tokenizer.is_fast})")

print("Loading training data...")
train_df = load_esnli_train(config)
print(f"  Train examples: {len(train_df)}")

print("Loading dev data...")
dev_df = load_esnli_split(config, "dev")
print(f"  Dev examples: {len(dev_df)}")

train_dataset = ESNLIMultiTaskDataset(train_df, tokenizer, config, is_train=True)
eval_dataset = ESNLIMultiTaskDataset(dev_df, tokenizer, config, is_train=False)

# Sanity check: inspect one example
sample = train_dataset[0]
print(f"\nSample shapes:")
for k, v in sample.items():
    print(f"  {k}: {v.shape}")
print(f"  MLM labels != -100: {(sample['mlm_labels'] != -100).sum().item()} tokens masked")

In [ ]:
# Cell 4: Initialize model
from models.variant_c import DeBERTaForMultiTask

model = DeBERTaForMultiTask(
    model_name=config.model_name,
    num_labels=config.num_labels,
    alpha=config.alpha,
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
# Cell 5: Check for existing checkpoint (for session recovery)
import glob

checkpoint_dir = str(config.get_path("checkpoint_dir"))
resume_from = None

existing_checkpoints = sorted(glob.glob(os.path.join(checkpoint_dir, "checkpoint-*")))
if existing_checkpoints:
    resume_from = existing_checkpoints[-1]
    print(f"Found checkpoint to resume from: {resume_from}")
else:
    print("No existing checkpoints found. Training from scratch.")

In [ ]:
# Cell 6: Train
import numpy as np
from transformers import TrainingArguments
from training.train import MultiTaskTrainer, compute_metrics

training_args = TrainingArguments(
    output_dir=checkpoint_dir,
    num_train_epochs=config.num_train_epochs,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,
    weight_decay=config.weight_decay,
    warmup_ratio=config.warmup_ratio,
    fp16=config.fp16,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=config.save_total_limit,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    dataloader_num_workers=config.dataloader_num_workers,
    logging_steps=100,
    report_to="none",
    remove_unused_columns=False,
)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train(resume_from_checkpoint=resume_from)

In [ ]:
# Cell 7: Save final model to Drive
import torch

final_dir = str(config.get_path("output_dir"))
os.makedirs(final_dir, exist_ok=True)
torch.save(model.state_dict(), os.path.join(final_dir, "model.pt"))
print(f"Model saved to {final_dir}/model.pt")

In [ ]:
# Cell 8: Evaluate on e-SNLI test (premise + hypothesis only — realistic inference)
from evaluation.evaluate import evaluate_dataset
from data.preprocess import NLIDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

esnli_test_df = load_esnli_split(config, "test")
esnli_test_dataset = NLIDataset(
    premises=esnli_test_df["Sentence1"].tolist(),
    hypotheses=esnli_test_df["Sentence2"].tolist(),
    labels=[config.label2id[lbl] for lbl in esnli_test_df["gold_label"].tolist()],
    tokenizer=tokenizer,
    max_length=config.max_seq_length,
)

acc = evaluate_dataset(model, esnli_test_dataset, batch_size=config.per_device_eval_batch_size, device=device)
print(f"e-SNLI test accuracy (no explanation at inference): {acc:.4f}")

In [ ]:
# Cell 9: Cross-domain evaluation (SNLI, MultiNLI, ANLI)
from data.preprocess import load_snli_test, load_multinli, load_anli
import json

results = {"esnli_test": acc}

# SNLI test
print("Evaluating on SNLI test...")
snli_test = load_snli_test(tokenizer, config.max_seq_length)
results["snli_test"] = evaluate_dataset(model, snli_test, config.per_device_eval_batch_size, device)
print(f"  SNLI test: {results['snli_test']:.4f}")

# MultiNLI
for split in ("validation_matched", "validation_mismatched"):
    print(f"Evaluating on MultiNLI {split}...")
    mnli = load_multinli(tokenizer, split=split, max_length=config.max_seq_length)
    key = f"multinli_{split.replace('validation_', '')}"
    results[key] = evaluate_dataset(model, mnli, config.per_device_eval_batch_size, device)
    print(f"  MultiNLI {split}: {results[key]:.4f}")

# ANLI
for r in ("r1", "r2", "r3"):
    print(f"Evaluating on ANLI {r}...")
    anli = load_anli(tokenizer, round_tag=r, max_length=config.max_seq_length)
    results[f"anli_{r}"] = evaluate_dataset(model, anli, config.per_device_eval_batch_size, device)
    print(f"  ANLI {r}: {results[f'anli_{r}']:.4f}")

print("\n--- All Results ---")
for k, v in results.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# Cell 10: Save results to Drive
results_path = os.path.join(final_dir, "eval_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {results_path}")